In [31]:
import pandas as pd
import numpy as np

In [32]:
path = 'dengue 2022.csv'
df = pd.read_csv(path, encoding='latin1', sep = ';')
df_agua = pd.read_excel('/content/agua_muni.xlsx')

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9517 entries, 0 to 9516
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ANO                      9517 non-null   int64  
 1   TIPO_ABOVIROSE           9517 non-null   object 
 2   SUPERINTENDENCIA         9517 non-null   object 
 3   ADS                      9517 non-null   object 
 4   MUNICÍPIO                9517 non-null   object 
 5   DATA_EPIDEMIOLOGICA      9517 non-null   object 
 6   SEMANA_EPIDEMIOLOGICA    9517 non-null   int64  
 7   DENGUE_GRAVE             9516 non-null   float64
 8   CURA                     9516 non-null   float64
 9   OBITO                    9516 non-null   float64
 10  OBITO_INVESTIGACAO       9516 non-null   float64
 11  CONFIRMADO_AMBULATORIAL  9516 non-null   float64
 12  CONFIRMADO_CLINICO       9516 non-null   float64
dtypes: float64(6), int64(2), object(5)
memory usage: 966.7+ KB


In [34]:
df.head()

,ANO,TIPO_ABOVIROSE,SUPERINTENDENCIA,ADS,MUNICÍPIO,DATA_EPIDEMIOLOGICA,SEMANA_EPIDEMIOLOGICA,DENGUE_GRAVE,CURA,OBITO,OBITO_INVESTIGACAO,CONFIRMADO_AMBULATORIAL,CONFIRMADO_CLINICO
0,2022,Dengue,SOBRAL,12ª REGIÃO ACARAÚ,BELA CRUZ,2022-01-29,4,0.0,0.0,0.0,0.0,0.0,0.0
1,2022,Dengue,SOBRAL,15ª REGIÃO CRATEÚS,NOVA RUSSAS,2022-12-10,49,0.0,0.0,0.0,0.0,0.0,0.0
2,2022,Dengue,FORTALEZA,03ª REGIÃO MARACANAÚ,GUAIUBA,2022-08-06,31,0.0,0.0,0.0,0.0,0.0,0.0
3,2022,Dengue,FORTALEZA,22ª REGIÃO CASCAVEL,CHOROZINHO,2022-04-16,15,0.0,0.0,0.0,0.0,0.0,0.0
4,2022,Dengue,LITORAL LESTE/JAGUARIBE,10ª REGIÃO LIMOEIRO DO NORTE,JAGUARIBE,2022-10-08,40,0.0,4.0,0.0,0.0,1.0,0.0


In [35]:
df['confirmado'] = df['CONFIRMADO_AMBULATORIAL'] + df['CONFIRMADO_CLINICO']

In [36]:
df_dengue = df.groupby('MUNICÍPIO')['confirmado'].sum()

In [37]:
df_agua['MUNICÍPIO']  = df_agua['MUNICÍPIO'].str.upper()

In [45]:
df_final = pd.merge(df_dengue, df_agua, on = 'MUNICÍPIO')

In [41]:
df_dengue

,confirmado
MUNICÍPIO,
ABAIARA,4.0
ACARAPE,5.0
ACARAU,22.0
ACOPIARA,23.0
AIUABA,220.0
...,...
URUBURETAMA,8.0
URUOCA,3.0
VARJOTA,3.0


In [42]:
df_agua

,MUNICÍPIO,Não possui ligação com a rede geral
0,ABAIARA,34.99
1,ACARAPE,12.32
2,ACARAÚ,23.65
3,ACOPIARA,31.14
4,AIUABA,35.61
...,...,...
179,URUBURETAMA,22.33
180,URUOCA,22.04
181,VARJOTA,7.68
182,VÁRZEA ALEGRE,21.18


In [52]:
df_final

,MUNICÍPIO,confirmado,Não possui ligação com a rede geral
0,ABAIARA,4.0,34.99
1,ACARAPE,5.0,12.32
2,ACARAU,22.0,23.65
3,ACOPIARA,23.0,31.14
4,AIUABA,220.0,35.61
...,...,...,...
179,URUBURETAMA,8.0,22.33
180,URUOCA,3.0,22.04
181,VARJOTA,3.0,7.68
182,VARZEA ALEGRE,26.0,21.18


In [49]:
import unicodedata

def remover_acentos(txt):
    if not isinstance(txt, str): return txt
    # Normaliza, remove acentos, transforma em maiúsculo e remove espaços extras
    return ''.join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn').upper().strip()

# Resetando o index do df_dengue
df_dengue_clean = df_dengue.reset_index()

# Aplicando a normalização robusta
df_dengue_clean['MUNICÍPIO'] = df_dengue_clean['MUNICÍPIO'].apply(remover_acentos)
df_agua['MUNICÍPIO'] = df_agua['MUNICÍPIO'].apply(remover_acentos)

# Realizando o merge
df_final = pd.merge(df_dengue_clean, df_agua, on='MUNICÍPIO', how='inner')

print(f'Total de municípios após merge: {len(df_final)}')
print('\nExemplos em df_dengue:', df_dengue_clean['MUNICÍPIO'].head().tolist())
print('Exemplos em df_agua:', df_agua['MUNICÍPIO'].head().tolist())

display(df_final.head())

Total de municípios após merge: 184

Exemplos em df_dengue: ['ABAIARA', 'ACARAPE', 'ACARAU', 'ACOPIARA', 'AIUABA']
Exemplos em df_agua: ['ABAIARA', 'ACARAPE', 'ACARAU', 'ACOPIARA', 'AIUABA']


,MUNICÍPIO,confirmado,Não possui ligação com a rede geral
0,ABAIARA,4.0,34.99
1,ACARAPE,5.0,12.32
2,ACARAU,22.0,23.65
3,ACOPIARA,23.0,31.14
4,AIUABA,220.0,35.61


In [50]:
df_final.tail()

,MUNICÍPIO,confirmado,Não possui ligação com a rede geral
179,URUBURETAMA,8.0,22.33
180,URUOCA,3.0,22.04
181,VARJOTA,3.0,7.68
182,VARZEA ALEGRE,26.0,21.18
183,VICOSA DO CEARA,53.0,36.69


In [51]:
from google.colab import files

df_final.to_excel('df_final.xlsx', index=False)
files.download('df_final.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>